# 05 Open-Meteo API and Kafka Producer

## Purpose
Build Open-Meteo requests, create internal air-quality events, and send them to a group-specific Kafka topic.

## Inputs
City reference coordinates and environment variables.

## Outputs
Kafka events on `KAFKA_TOPIC_AIR_QUALITY_LIVE` and optional tiny raw API samples.

## Technologies used
Python, requests, kafka-python, JSON.

## Configuration
Producer execution is guarded by `RUN_OPEN_METEO_KAFKA_PRODUCER = False`. Kafka settings come from `.env`.

In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd()
DATA_DIR = Path(os.getenv("DATA_DIR", "data"))
CHECKPOINT_DIR = Path(os.getenv("CHECKPOINT_DIR", "data/checkpoints"))


## Implementation
Create events with `city_id`, timestamp, pollutants, source, and schema version. Do not use generic Kafka topics or hardcoded credentials.

In [ ]:
import json, os
RUN_OPEN_METEO_KAFKA_PRODUCER = False
TOPIC = os.getenv('KAFKA_TOPIC_AIR_QUALITY_LIVE', 'bdeng_gXX_air_quality_live')

def build_air_quality_event(city_id, timestamp_utc, hourly_row):
    return {'schema_version': '1.0', 'source': 'open_meteo', 'city_id': city_id, 'timestamp_utc': timestamp_utc, 'pollutants': {'pm2_5': hourly_row.get('pm2_5'), 'pm10': hourly_row.get('pm10'), 'nitrogen_dioxide': hourly_row.get('nitrogen_dioxide')}}
TOPIC

## Validation / Quality Checks
Validate schema keys and reject generic shared topic names.

In [ ]:
event = build_air_quality_event('vienna_at', '2026-05-30T00:00:00Z', {'pm2_5': 5, 'pm10': 12, 'nitrogen_dioxide': 20})
assert event['pollutants']['pm2_5'] == 5
assert TOPIC != 'air_quality_live'
print('Open-Meteo event schema smoke check passed')

## Results
The notebook provides the Kafka producer implementation path without running it during refactor.

## Limitations
Live API values are current context and must not be mixed with historical EEA conclusions without labeling.

## Next step
Run notebook `06` for Spark Structured Streaming from Kafka to Parquet.